<a href="https://colab.research.google.com/github/jgromero/drl-csic/blob/es/code/from-scratch/cliffwalking-dqn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Deep Q-Learning para _cliff walking_

[**Juan Gómez Romero**](https://ccia.ugr.es/~jgomez)  
Departamento de Ciencias de la Computación e Inteligencia Artificial  
Universidad de Granada  
Este trabajo está bajo licencia [Apache License 2.0](https://choosealicense.com/licenses/apache-2.0/).

---
Basado en:
> R.S. Sutton, A.G. Barto (2018) Reinforcement Learning. Capítulos 6.5: Q-learning: control TD _off-policy_, 6.6: _Expected Sarsa_.

> Udacity, Deep Reinforcement Learning Course. Disponible en [GitHub](https://github.com/udacity/deep-reinforcement-learning/tree/master/dqn).

## Configuración

In [ ]:
%pip install "gymnasium[toy-text]>=1.1"

In [ ]:
# modo de renderizado 'rgb_array': el entorno genera imágenes que se muestran como animaciones en el cuaderno
render_mode = "rgb_array"

In [ ]:
# comprobar si existen los ficheros de apoyo
import os.path
if not os.path.isfile('dqn_agent.py'):
  !git clone -b es https://github.com/jgromero/drl-csic
  %cd drl-csic/code/from-scratch

## Entorno

Usaremos [CliffWalking-v1](https://gymnasium.farama.org/environments/toy_text/cliff_walking/), el mismo entorno que resolvimos con Q-Learning tabular en [cliffwalking-qlearning.ipynb](https://github.com/jgromero/drl-csic/blob/es/code/from-scratch/cliffwalking-qlearning.ipynb).

_Considere el mundo de cuadrícula ("gridworld") mostrado a la derecha. Es una tarea episódica estándar, sin descuento, con estados inicial y meta, y con las acciones habituales que provocan un movimiento hacia arriba, abajo, derecha e izquierda. La recompensa es -1 en todas las transiciones, salvo en las que llevan a la región marcada como «El Acantilado» ("The Cliff"). Pisar esta región supone una recompensa de -100 y devuelve al agente instantáneamente al estado inicial._

<img src="https://github.com/jgromero/drl-csic/blob/es/img/cliffwalking.png?raw=true" width=500/>

Queremos reutilizar, sin ningún cambio, el mismo agente DQN que usamos para [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/). Esto requiere dos adaptaciones del entorno:

*   La observación de CliffWalking es el índice de la celda en la que se encuentra el agente (un entero de $0$ a $47$), pero la red Q espera un vector de números. Transformamos cada observación en un vector _one-hot_ de tamaño $48$ (todo ceros, salvo un $1$ en la posición de la celda) con el _wrapper_ [`TransformObservation`](https://gymnasium.farama.org/api/wrappers/observation_wrappers/#gymnasium.wrappers.TransformObservation).
*   Caer por el acantilado no termina el episodio, así que una mala política puede caminar indefinidamente. Limitamos la longitud de los episodios a $100$ pasos con el parámetro `max_episode_steps`.

Los espacios de acciones y de observaciones se pueden mostrar como sigue.

In [ ]:
import gymnasium as gym
import numpy as np
from gymnasium.wrappers import TransformObservation

def one_hot(state):
    """Codifica el índice de una celda (0-47) como un vector one-hot de tamaño 48."""
    return np.eye(48, dtype=np.float32)[state]

env = gym.make("CliffWalking-v1", render_mode = render_mode, max_episode_steps=100)
env = TransformObservation(env, one_hot, gym.spaces.Box(0, 1, (48,), np.float32))
print(env.observation_space)
print(env.action_space)

Para ver cómo actúa un agente, la función `show_episode()` ejecuta un episodio completo y muestra las imágenes generadas por el entorno (`render_mode = "rgb_array"`) como una animación, tanto en Google Colab como en local. La usaremos para comparar el comportamiento del agente antes y después del entrenamiento.

In [ ]:
import io
from PIL import Image
from IPython.display import display, Image as IPImage

def show_episode(env, act, max_t=500):
    """Ejecuta un episodio y lo muestra en el cuaderno como una animación.

    Parámetros
    ==========
        env: entorno creado con render_mode="rgb_array"
        act (callable): función que devuelve la acción para un estado
        max_t (int): número máximo de pasos del episodio
    """
    state, _ = env.reset()
    frames = [Image.fromarray(env.render())]
    score = 0
    for t in range(max_t):
        state, reward, terminated, truncated, _ = env.step(act(state))
        frames.append(Image.fromarray(env.render()))
        score += reward
        if terminated or truncated:
            break

    # guardar las imágenes como un GIF animado y mostrarlo
    gif = io.BytesIO()
    frames[0].save(gif, format='gif', save_all=True, append_images=frames[1:],
                   duration=1000 // env.metadata['render_fps'], loop=0)
    display(IPImage(data=gif.getvalue(), format='gif'))
    print('Puntuación:', score)

Podemos interactuar con el entorno con un agente sin entrenar usando la clase `Agent` de [dqn_agent.py](https://github.com/jgromero/drl-csic/blob/es/code/from-scratch/dqn_agent.py). La única diferencia con CartPole es el tamaño del estado ($48$) y el número de acciones ($4$).

In [ ]:
from dqn_agent import Agent

agent_untrained = Agent(state_size=48, action_size=4, seed=0)

show_episode(env, agent_untrained.act, max_t=100)

## Algoritmo DQN
A continuación usamos la misma implementación de Deep Q-Learning (DQN) que en [cartpole-dqn.ipynb](https://github.com/jgromero/drl-csic/blob/es/code/from-scratch/cartpole-dqn.ipynb). Nos apoyamos en ficheros externos:

*   [dqn_agent.py](https://github.com/jgromero/drl-csic/blob/es/code/from-scratch/dqn_agent.py): Implementación de las capacidades del agente (`act`, `step`, etc.)
*   [model.py](https://github.com/jgromero/drl-csic/blob/es/code/from-scratch/model.py): Implementación de la red neuronal del agente (la red Q)

La política óptima (caminar por el borde del acantilado) obtiene una puntuación de $-13$. Sin embargo, durante el entrenamiento el agente sigue tomando algunas acciones aleatorias ($\epsilon$-_greedy_), que ocasionalmente le hacen caer por el acantilado. Por ello, consideramos el problema resuelto cuando el agente obtiene una puntuación media de al menos $-20$ puntos en 100 episodios consecutivos.

In [ ]:
import random
import torch
import numpy as np
from collections import deque
from dqn_agent import device
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
def plot_scores(scores, title, solved_score=None, window=100):
    """Representa la puntuación de cada episodio de entrenamiento y la media móvil de los últimos `window`.

    Parámetros
    ==========
        scores (list): puntuación de cada episodio de entrenamiento
        title (str): título de la gráfica
        solved_score (float): puntuación media para considerar resuelto el problema (opcional)
        window (int): número de episodios de la media móvil
    """
    scores = np.asarray(scores)
    avg = [np.mean(scores[max(0, i-window+1):i+1]) for i in range(len(scores))]
    plt.plot(np.arange(len(scores)), scores, alpha=0.4, label='Puntuación')
    plt.plot(np.arange(len(scores)), avg, label='Puntuación media (últimos {})'.format(window))
    if solved_score is not None:
        plt.axhline(solved_score, color='gray', linestyle='--', label='Resuelto')
    plt.ylabel('Puntuación')
    plt.xlabel('Episodio nº')
    plt.title(title)
    plt.legend()
    plt.show()

In [ ]:
SOLVED_SCORE = -20.0   # puntuación media (últimos 100 episodios) para considerar resuelto CliffWalking-v1

def dqn(agent, n_episodes=2000, max_t=100, eps_start=1.0, eps_end=0.01, eps_decay=0.995, print_every=100):
    """Deep Q-Learning.

    Parámetros
    ==========
        agent (Agent): agente a entrenar
        n_episodes (int): número máximo de episodios
        max_t (int): número máximo de pasos por episodio
        eps_start (float): valor inicial de epsilon
        eps_end (float): valor final de epsilon
        eps_decay (float): factor de multiplicación de epsilon (por episodio)
        print_every (int): mostrar la puntuación cada print_every episodios
    """
    scores = []                        # puntuaciones de todos los episodios
    scores_window = deque(maxlen=100)  # puntuaciones de los últimos 100 episodios
    eps = eps_start                    # inicializar epsilon

    for i_episode in range(1, n_episodes+1):
        state, _ = env.reset()
        score = 0

        for t in range(max_t):

            # elegir la acción At con la política ε-greedy
            action = agent.act(state, eps)

            # aplicar At y obtener Rt+1, St+1
            next_state, reward, terminated, truncated, _ = env.step(action)

            # almacenar <St, At, Rt+1, St+1>, entrenar y actualizar
            # (solo 'terminated' marca St+1 como terminal; tras un truncamiento hay que seguir haciendo bootstrapping)
            agent.step(state, action, reward, next_state, terminated)

            # avanzar al siguiente estado
            state = next_state
            score += reward

            if terminated or truncated:
                break

        scores_window.append(score)       # guardar la última puntuación (ventana de 100)
        scores.append(score)              # guardar la última puntuación (todos los episodios)
        eps = max(eps_end, eps_decay*eps) # reducir epsilon

        # mostrar información del entrenamiento
        if i_episode % print_every == 0:
            print('Episodio {}\tPuntuación media (últimos 100): {:.2f}'.format(i_episode, np.mean(scores_window)))
        if len(scores_window) == 100 and np.mean(scores_window) >= SOLVED_SCORE:
            print('\n¡Problema resuelto en {:d} episodios!\tPuntuación media (últimos 100): {:.2f}'.format(i_episode-100, np.mean(scores_window)))
            break
    else:
        print('\nProblema no resuelto en {:d} episodios.\tPuntuación media (últimos 100): {:.2f}'.format(n_episodes, np.mean(scores_window)))

    torch.save(agent.qnetwork_local.state_dict(), 'checkpoint-cliffwalking-dqn.pth') # guardar los pesos del agente entrenado

    return scores

agent = Agent(state_size=48, action_size=4, seed=0)
scores = dqn(agent)

plot_scores(scores, 'DQN', SOLVED_SCORE)

## Política aprendida y función de valor de estado

Como en [cliffwalking-qlearning.ipynb](https://github.com/jgromero/drl-csic/blob/es/code/from-scratch/cliffwalking-qlearning.ipynb), podemos mostrar la política resultante (_greedy_). Aquí los valores $Q(s, a)$ no se leen de una tabla, sino que los calcula la red Q (`agent.qnetwork_local`) para los vectores _one-hot_ de los 48 estados.

In [ ]:
def print_policy(actions):
    """Muestra la política como una cuadrícula de flechas (C: acantilado, T: meta).

    Parámetros
    ==========
        actions (array_like): acción de cada uno de los 48 estados (ARRIBA = 0, DERECHA = 1, ABAJO = 2, IZQUIERDA = 3)
    """
    grid = np.array(['↑', '→', '↓', '←'])[np.asarray(actions)].reshape(4, 12)
    grid[3, 1:11] = 'C'   # celdas del acantilado
    grid[3, 11] = 'T'     # meta
    print('\n'.join(' '.join(row) for row in grid))

In [ ]:
states = torch.eye(48).to(device)   # vectores one-hot de los 48 estados
with torch.no_grad():
    Q = agent.qnetwork_local(states).cpu().numpy()   # valores Q de los 48 estados, forma (48, 4)

print_policy(Q.argmax(axis=1))

Podemos visualizar los valores de la función de valor de estado $v_\pi$ para la política estimada $\pi$ y compararlos con los valores de la política óptima $v_{\pi_*}$. Nótese que [dqn_agent.py](https://github.com/jgromero/drl-csic/blob/es/code/from-scratch/dqn_agent.py) usa $\gamma = 0.99$ en lugar de $\gamma = 1$, por lo que los valores de los estados lejanos a la meta son ligeramente mayores (menos negativos) que los óptimos.

(Se necesita la función `plot_values()` para la visualización.)

In [ ]:
def plot_values(V):
    # redimensionar la función de valor de estado
    V = np.reshape(V, (4,12))
    # representar la función de valor de estado
    fig = plt.figure(figsize=(15,5))
    ax = fig.add_subplot(111)
    im = ax.imshow(V, cmap='cool')
    for (j,i),label in np.ndenumerate(V):
        ax.text(i, j, '{:.1f}'.format(label), ha='center', va='center', fontsize=14)
    plt.tick_params(bottom=False, left=False, labelbottom=False, labelleft=False)
    plt.title('Función de valor de estado')
    plt.show()

__Función de valor de estado de la política aprendida por el agente $v_\pi$__

In [ ]:
V = Q.max(axis=1)
V[37:] = 0   # las celdas del acantilado y de la meta nunca se visitan
plot_values(V)

__Función de valor de estado de la política óptima $v_{\pi_*}$__

In [ ]:
# Los valores de estado de la política óptima pueden calcularse manualmente
V_opt = np.zeros((4,12))
V_opt[0][:] = -np.arange(3, 15)[::-1]
V_opt[1][:] = -np.arange(3, 15)[::-1] + 1
V_opt[2][:] = -np.arange(3, 15)[::-1] + 2
V_opt[3][0] = -13
plot_values(V_opt)

## Visualizar el agente entrenado

Cargamos los pesos en el agente y lo ejecutamos durante un episodio con la política _greedy_. Las imágenes se muestran como una animación, tanto en Google Colab como en local.

In [ ]:
# cargar los pesos desde `checkpoint-cliffwalking-dqn.pth`
agent_load = Agent(state_size=48, action_size=4, seed=0)

agent_load.qnetwork_local.load_state_dict(torch.load('checkpoint-cliffwalking-dqn.pth', map_location=device, weights_only=True))

show_episode(env, agent_load.act, max_t=100)